In [2]:
from langgraph.graph import StateGraph, END
import random 
from typing import Dict, List, TypedDict

In [3]:
class GameState(TypedDict):
    player_name: str 
    target_number: int 
    guesses: List[int]
    attempts: int 
    hint: str 
    lower_bound: int 
    upper_bound: int

In [4]:
def setup_node(state: GameState) -> GameState:
    """
        Initialize the game with a random target number
    """
    state["player_name"] = f"Welcome, {state["player_name"]}"
    state["target_number"] = random.randint(1,20)
    state["guesses"] = []
    state["attempts"] = 0
    state["hint"] = "Game started! Try to guess the number"
    state["lower_bound"] = 1
    state["upper_bound"] = 20
    print(f"{state['player_name']} The game has begun. I'm thinking of a number between 1 and 20.")
    return state

In [5]:
def guess_node(state: GameState) -> GameState:
    """Generate a smarter guess based on previous hints"""
    
    possible_guesses = [i for i in range(state["lower_bound"], state["upper_bound"] + 1) if i not in state["guesses"]]
    if possible_guesses:
        guess = random.choice(possible_guesses)
    else:
        
        guess = random.randint(state["lower_bound"], state["upper_bound"])
    
    state["guesses"].append(guess)
    state["attempts"] += 1
    print(f"Attempt {state['attempts']}: Guessing {guess} (Current range: {state['lower_bound']}-{state['upper_bound']})")
    return state

In [6]:
def hint_node(state: GameState) -> GameState:
    """Here we provide a hint based on the last guess and update the bounds"""
    latest_guess = state["guesses"][-1]
    target = state["target_number"]
    
    if latest_guess < target:
        state["hint"] = f"The number {latest_guess} is too low. Try higher!"
        
        state["lower_bound"] = max(state["lower_bound"], latest_guess + 1)
        print(f"Hint: {state['hint']}")
        
    elif latest_guess > target:
        state["hint"] = f"The number {latest_guess} is too high. Try lower!"
      
        state["upper_bound"] = min(state["upper_bound"], latest_guess - 1)
        print(f"Hint: {state['hint']}")
    else:
        state["hint"] = f"Correct! You found the number {target} in {state['attempts']} attempts."
        print(f"Success! {state['hint']}")
    
    return state

In [7]:
def should_continue(state: GameState) -> str:
    """Determine if we should continue guessing or end the game"""
    
    # There are 2 end conditions - either 7 is reached or the correct number is guessed
    
    latest_guess = state["guesses"][-1]
    if latest_guess == state["target_number"]:
        print(f"GAME OVER: Number found!")
        return "end"
    elif state["attempts"] >= 7:
        print(f"GAME OVER: Maximum attempts reached! The number was {state['target_number']}")
        return "end"
    else:
        print(f"CONTINUING: {state['attempts']}/7 attempts used")
        return "continue"

In [8]:
graph = StateGraph(GameState)
graph.add_node("setup", setup_node)
graph.add_node("guess", guess_node)
graph.add_node("hint_node", hint_node)  


graph.add_edge("setup", "guess")
graph.add_edge("guess", "hint_node")  


graph.add_conditional_edges(
    "hint_node", 
    should_continue,
    {
        "continue": "guess",
        "end": END
    }
)

graph.set_entry_point("setup")
app = graph.compile()

In [11]:
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))

ValueError: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

In [17]:
result = app.invoke({"player_name": "Student", "guesses": [], "attempts": 0, "lower_bound": 1, "upper_bound": 20})

Welcome, Student The game has begun. I'm thinking of a number between 1 and 20.
Attempt 1: Guessing 18 (Current range: 1-20)
Hint: The number 18 is too high. Try lower!
CONTINUING: 1/7 attempts used
Attempt 2: Guessing 8 (Current range: 1-17)
Hint: The number 8 is too low. Try higher!
CONTINUING: 2/7 attempts used
Attempt 3: Guessing 17 (Current range: 9-17)
Hint: The number 17 is too high. Try lower!
CONTINUING: 3/7 attempts used
Attempt 4: Guessing 14 (Current range: 9-16)
Hint: The number 14 is too high. Try lower!
CONTINUING: 4/7 attempts used
Attempt 5: Guessing 12 (Current range: 9-13)
Hint: The number 12 is too high. Try lower!
CONTINUING: 5/7 attempts used
Attempt 6: Guessing 9 (Current range: 9-11)
Hint: The number 9 is too low. Try higher!
CONTINUING: 6/7 attempts used
Attempt 7: Guessing 11 (Current range: 10-11)
Hint: The number 11 is too high. Try lower!
GAME OVER: Maximum attempts reached! The number was 10
